# ITO5202 Assessment 2 — Machine Learning and Real-Time Streaming

**Student ID:** 29701201  **Unit:** ITO5202
**Dataset:** Brazilian E-Commerce Public Dataset by Olist ([Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce))


## Environment Setup

The Spark configuration is carried over from Assessment 1, so results from the two notebooks are directly comparable:

- `local[*]` uses every core the Docker container exposes.
- `spark.driver.memory = 4g` must be set before the JVM starts, so this cell must be the first Spark call after a kernel restart.
- `spark.sql.session.timeZone = UTC`. Olist timestamps are Brazilian local times with no zone attached. Parsing them in UTC stops the container's time zone from shifting them, which would change delivery durations and the late-delivery flag.
- `spark.sql.shuffle.partitions = 2 × cores`. The order-level table is about 96k rows, so the default 200 shuffle partitions would create mostly empty tasks.

In [1]:
# ---------------------------------------------------------------
# Imports
# ---------------------------------------------------------------
import os          # file paths and CPU count
import platform    # Python / OS version for the environment table
import shutil      # removing old output folders before re-writing them

import pandas as pd

# Spark configuration and entry points
from pyspark import SparkConf
from pyspark.sql import SparkSession

# DataFrame functions, window specifications and schema types
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType, TimestampType)

In [2]:
# ---------------------------------------------------------------
# Spark configuration
# ---------------------------------------------------------------
master = "local[*]"
app_name = "ITO5202_A2_Olist_29701201"

spark_conf = (
    SparkConf()
    .setMaster(master)
    .setAppName(app_name)
    # In local mode the driver is also the executor, so this is all the
    # memory Spark has. Only takes effect when the JVM starts.
    .set("spark.driver.memory", "4g")
    # Parse timestamps exactly as written in the CSVs (no time-zone shifting)
    .set("spark.sql.session.timeZone", "UTC")
    # Hide console progress bars so the PDF export stays clean
    .set("spark.ui.showConsoleProgress", "false")
)

spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")  # errors only, to keep outputs readable

# 2 x cores
SHUFFLE_PARTITIONS = sc.defaultParallelism * 2
spark.conf.set("spark.sql.shuffle.partitions", SHUFFLE_PARTITIONS)

print(f"Spark {spark.version} session started: {app_name}")
print(f"Shuffle partitions set to {SHUFFLE_PARTITIONS} "
      f"({sc.defaultParallelism} cores x 2)")

Spark 4.1.1 session started: ITO5202_A2_Olist_29701201
Shuffle partitions set to 16 (8 cores x 2)


In [3]:
# ---------------------------------------------------------------
# Execution environment summary
# ---------------------------------------------------------------
env = {
    "Execution mode": f"local ({sc.master})",
    "Spark version": spark.version,
    "Python version": platform.python_version(),
    "OS": f"{platform.system()} {platform.release()}",
    "CPU cores visible": os.cpu_count(),
    "defaultParallelism": sc.defaultParallelism,
    "Driver memory": spark.conf.get("spark.driver.memory", "default (1g)"),
    "spark.sql.shuffle.partitions": spark.conf.get("spark.sql.shuffle.partitions"),
    "Session time zone": spark.conf.get("spark.sql.session.timeZone"),
    "Spark Web UI": "http://localhost:4040",
}
pd.DataFrame(env.items(), columns=["Setting", "Value"])

,Setting,Value
0,Execution mode,local (local[*])
1,Spark version,4.1.1
2,Python version,3.13.12
3,OS,Linux 7.0.12-linuxkit
4,CPU cores visible,8
5,defaultParallelism,8
6,Driver memory,4g
7,spark.sql.shuffle.partitions,16
8,Session time zone,UTC
9,Spark Web UI,http://localhost:4040


## Data Loading

### Tables used
Assessment 1 used six tables. Assessment 2 adds three, because they hold information that is known when an order is placed and that plausibly affects delivery:

| Table | Used for | New in A2? |
|---|---|---|
| `orders` | Purchase, approval, estimated and actual delivery timestamps | |
| `order_items` | Price, freight, seller and product for each line item | |
| `customers` | Customer state and zip-code prefix | |
| `products` | Category, weight and dimensions | |
| `category_translation` | English category names | |
| `order_reviews` | Review score (kept as an outcome column only) | |
| `order_payments` | Payment type and number of instalments | **Yes** |
| `sellers` | Seller state and zip-code prefix | **Yes** |
| `geolocation` | Latitude/longitude per zip-code prefix, for customer–seller distance | **Yes** |

In A1, `order_payments` was excluded because joining per-order payments to line items would double-count revenue. Here that problem does not arise: payments and items are each **aggregated to one row per order before they are joined** 

In [4]:
# ---------------------------------------------------------------
# File locations
# ---------------------------------------------------------------
# CSVs sit in ./data/ next to this notebook (see data/README.md).
DATA_DIR = "data"

FILES = {
    "orders":               "olist_orders_dataset.csv",
    "order_items":          "olist_order_items_dataset.csv",
    "customers":            "olist_customers_dataset.csv",
    "products":             "olist_products_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
    "order_reviews":        "olist_order_reviews_dataset.csv",
    "order_payments":       "olist_order_payments_dataset.csv",
    "sellers":              "olist_sellers_dataset.csv",
    "geolocation":          "olist_geolocation_dataset.csv",
}

missing = [f for f in FILES.values() if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    raise FileNotFoundError(f"Missing from ./{DATA_DIR}/: {missing}. See data/README.md.")
print("All data files found.")

All data files found.


In [5]:
# ---------------------------------------------------------------
# Explicit schemas
# ---------------------------------------------------------------
# Column ORDER must match each CSV exactly (Spark maps columns by position).
# Zip-code prefixes are strings so leading zeros are kept.
# Olist's own header misspells 'length' as 'lenght' in two product columns.
SCHEMAS = {
    "orders": StructType([
        StructField("order_id", StringType(), True),
        StructField("customer_id", StringType(), True),
        StructField("order_status", StringType(), True),
        StructField("order_purchase_timestamp", TimestampType(), True),
        StructField("order_approved_at", TimestampType(), True),
        StructField("order_delivered_carrier_date", TimestampType(), True),
        StructField("order_delivered_customer_date", TimestampType(), True),
        StructField("order_estimated_delivery_date", TimestampType(), True),
    ]),
    "order_items": StructType([
        StructField("order_id", StringType(), True),
        StructField("order_item_id", IntegerType(), True),
        StructField("product_id", StringType(), True),
        StructField("seller_id", StringType(), True),
        StructField("shipping_limit_date", TimestampType(), True),
        StructField("price", DoubleType(), True),
        StructField("freight_value", DoubleType(), True),
    ]),
    "customers": StructType([
        StructField("customer_id", StringType(), True),
        StructField("customer_unique_id", StringType(), True),
        StructField("customer_zip_code_prefix", StringType(), True),
        StructField("customer_city", StringType(), True),
        StructField("customer_state", StringType(), True),
    ]),
    "products": StructType([
        StructField("product_id", StringType(), True),
        StructField("product_category_name", StringType(), True),
        StructField("product_name_lenght", IntegerType(), True),
        StructField("product_description_lenght", IntegerType(), True),
        StructField("product_photos_qty", IntegerType(), True),
        StructField("product_weight_g", DoubleType(), True),
        StructField("product_length_cm", DoubleType(), True),
        StructField("product_height_cm", DoubleType(), True),
        StructField("product_width_cm", DoubleType(), True),
    ]),
    "category_translation": StructType([
        StructField("product_category_name", StringType(), True),
        StructField("product_category_name_english", StringType(), True),
    ]),
    "order_reviews": StructType([
        StructField("review_id", StringType(), True),
        StructField("order_id", StringType(), True),
        StructField("review_score", IntegerType(), True),
        StructField("review_comment_title", StringType(), True),
        StructField("review_comment_message", StringType(), True),
        StructField("review_creation_date", TimestampType(), True),
        StructField("review_answer_timestamp", TimestampType(), True),
    ]),
    "order_payments": StructType([
        StructField("order_id", StringType(), True),
        StructField("payment_sequential", IntegerType(), True),
        StructField("payment_type", StringType(), True),
        StructField("payment_installments", IntegerType(), True),
        StructField("payment_value", DoubleType(), True),
    ]),
    "sellers": StructType([
        StructField("seller_id", StringType(), True),
        StructField("seller_zip_code_prefix", StringType(), True),
        StructField("seller_city", StringType(), True),
        StructField("seller_state", StringType(), True),
    ]),
    "geolocation": StructType([
        StructField("geolocation_zip_code_prefix", StringType(), True),
        StructField("geolocation_lat", DoubleType(), True),
        StructField("geolocation_lng", DoubleType(), True),
        StructField("geolocation_city", StringType(), True),
        StructField("geolocation_state", StringType(), True),
    ]),
}

In [6]:
# ---------------------------------------------------------------
# Header validation and CSV loading 
# ---------------------------------------------------------------
def check_header(name):
    '''Compare the CSV header row with the schema's field names.'''
    path = os.path.join(DATA_DIR, FILES[name])
    header = spark.read.text(path).limit(1).first()[0]
    # Remove quote marks and the byte-order mark (﻿) some files start with
    file_cols = [c.strip().strip('"').lstrip("﻿") for c in header.split(",")]
    schema_cols = SCHEMAS[name].fieldNames()
    if file_cols != schema_cols:
        raise ValueError(f"{name}: header {file_cols} does not match schema {schema_cols}")


def load_csv(name, multiline=False):
    '''Load one Olist CSV using its explicit schema.'''
    check_header(name)
    reader = (
        spark.read
        .option("header", True)
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
        .option("mode", "PERMISSIVE")                     # bad values -> null
    )
    if multiline:
        # Review comments contain line breaks and quotes inside quoted fields
        reader = reader.option("multiLine", True).option("escape", '"')
    return reader.schema(SCHEMAS[name]).csv(os.path.join(DATA_DIR, FILES[name]))


orders_df               = load_csv("orders")
order_items_df          = load_csv("order_items")
customers_df            = load_csv("customers")
products_df             = load_csv("products")
category_translation_df = load_csv("category_translation")
order_reviews_df        = load_csv("order_reviews", multiline=True)
order_payments_df       = load_csv("order_payments")
sellers_df              = load_csv("sellers")
geolocation_df          = load_csv("geolocation")

tables = {
    "orders": orders_df, "order_items": order_items_df, "customers": customers_df,
    "products": products_df, "category_translation": category_translation_df,
    "order_reviews": order_reviews_df, "order_payments": order_payments_df,
    "sellers": sellers_df, "geolocation": geolocation_df,
}
rows = [(name, df.count(), len(df.columns)) for name, df in tables.items()]
pd.DataFrame(rows, columns=["Table", "Rows loaded", "Columns"])

,Table,Rows loaded,Columns
0,orders,99441,8
1,order_items,112650,7
2,customers,99441,5
3,products,32951,9
4,category_translation,71,2
5,order_reviews,99224,7
6,order_payments,103886,5
7,sellers,3095,4
8,geolocation,1000163,5


In [8]:
# ---------------------------------------------------------------
# Null counts per column, all nine tables
# ---------------------------------------------------------------

A2_USAGE = {
    "order_approved_at":             "Input (approval_hours)",
    "order_delivered_carrier_date":  "Not used (known only after dispatch)",
    "order_delivered_customer_date": "Outcome (is_late, delay_days) + population filter",
    "product_category_name":         "Input (main_category)",
    "product_name_lenght":           "Not used",
    "product_description_lenght":    "Not used",
    "product_photos_qty":            "Input (main_product_photos)",
    "product_weight_g":              "Input (total_weight_g)",
    "product_length_cm":             "Input (total_volume_cm3)",
    "product_height_cm":             "Input (total_volume_cm3)",
    "product_width_cm":              "Input (total_volume_cm3)",
    "review_comment_title":          "Not used (free text)",
    "review_comment_message":        "Not used (free text)",
}

def null_profile(name, df):
    '''One pass over the table: count nulls in every column.
    Only columns with at least one null are returned.'''
    n_rows = df.count()
    counts = df.select(
        [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
    ).first()
    return [(name, c, counts[c], round(100 * counts[c] / n_rows, 2),
             A2_USAGE.get(c, "-"))
            for c in df.columns if counts[c]]

profile = [r for name, df in tables.items() for r in null_profile(name, df)]
clean_tables = [name for name in tables if name not in {r[0] for r in profile}]

print(f"Tables with no nulls: {', '.join(clean_tables)}")
pd.DataFrame(profile, columns=["Table", "Column", "Null count", "% of rows", "Use in A2"])

Tables with no nulls: order_items, customers, category_translation, order_payments, sellers, geolocation


,Table,Column,Null count,% of rows,Use in A2
0,orders,order_approved_at,160,0.16,Input (approval_hours)
1,orders,order_delivered_carrier_date,1783,1.79,Not used (known only after dispatch)
2,orders,order_delivered_customer_date,2965,2.98,"Outcome (is_late, delay_days) + population filter"
3,products,product_category_name,610,1.85,Input (main_category)
4,products,product_name_lenght,610,1.85,Not used
5,products,product_description_lenght,610,1.85,Not used
6,products,product_photos_qty,610,1.85,Input (main_product_photos)
7,products,product_weight_g,2,0.01,Input (total_weight_g)
8,products,product_length_cm,2,0.01,Input (total_volume_cm3)
9,products,product_height_cm,2,0.01,Input (total_volume_cm3)


## Train/Stream Split Data


### Population
- purchased from 2017 Q1 to 2018 Q3 (A1 found the quarters outside this range too sparse: 4, 325 and 4 orders);
- status `delivered` with a recorded delivery date, because only these orders have a known outcome to learn from;
- at least one line item.
- orders without a review are kept


| Group | Columns | Known when? |
|---|---|---|
| Identifiers / time | `order_id`, `order_purchase_timestamp` | At purchase |
| Customer | `customer_state` | At purchase |
| Basket (aggregated over the order's items) | `n_items`, `n_sellers`, `total_price`, `total_freight`, `total_weight_g`, `total_volume_cm3`, `main_category`, `main_seller_state`, `main_product_photos` | At purchase |
| Payment (aggregated over payment records) | `payment_type`, `payment_installments`, `n_payment_methods`, `payment_value` | At purchase |
| Logistics promises | `estimated_delivery_days` (promised date − purchase), `shipping_limit_days` (seller's dispatch deadline − purchase), `approval_hours` | At purchase / approval |
| Geography | `distance_km` (customer ↔ main seller), `same_state` | At purchase |
| **Outcomes (never used as inputs)** | `delivery_days`, `delay_days`, `is_late`, `review_score` | Only after delivery |


In [9]:
# ---------------------------------------------------------------
# Population: delivered orders, 2017 Q1 - 2018 Q3 (as in A1)
# ---------------------------------------------------------------
START_TS = "2017-01-01 00:00:00"      # inclusive: start of 2017 Q1
END_TS   = "2018-10-01 00:00:00"      # exclusive: end of 2018 Q3

orders_pop_df = (
    orders_df
    .filter(
        (F.col("order_status") == "delivered") &
        F.col("order_delivered_customer_date").isNotNull() &
        (F.col("order_purchase_timestamp") >= F.to_timestamp(F.lit(START_TS))) &
        (F.col("order_purchase_timestamp") <  F.to_timestamp(F.lit(END_TS)))
    )
    .select("order_id", "customer_id", "order_purchase_timestamp", "order_approved_at",
            "order_delivered_customer_date", "order_estimated_delivery_date")
)
print(f"Orders in population: {orders_pop_df.count():,}")

Orders in population: 96,203


In [10]:
# ---------------------------------------------------------------
# Zip-code prefix -> one (lat, lng) centroid
# ---------------------------------------------------------------
# The geolocation table has ~1M rows: many points per prefix, plus a few
# coordinates that fall outside Brazil (data-entry errors). Points outside
# Brazil's bounding box are dropped, then the rest are averaged per prefix.
BRAZIL_LAT = (-33.75, 5.30)
BRAZIL_LNG = (-73.99, -34.79)

zip_centroid_df = (
    geolocation_df
    .filter(F.col("geolocation_lat").between(*BRAZIL_LAT) &
            F.col("geolocation_lng").between(*BRAZIL_LNG))
    .groupBy(F.col("geolocation_zip_code_prefix").alias("zip_prefix"))
    .agg(F.avg("geolocation_lat").alias("lat"),
         F.avg("geolocation_lng").alias("lng"))
)
n_raw = geolocation_df.count()
n_zip = zip_centroid_df.count()
print(f"{n_raw:,} geolocation rows reduced to {n_zip:,} zip-prefix centroids")

1,000,163 geolocation rows reduced to 19,010 zip-prefix centroids


In [11]:
# ---------------------------------------------------------------
# Items -> one row per order
# ---------------------------------------------------------------
# Per-item product volume and English category, then aggregate per order.
items_enriched_df = (
    order_items_df
    .join(F.broadcast(products_df), "product_id", "left")
    .join(F.broadcast(category_translation_df), "product_category_name", "left")
    .join(F.broadcast(sellers_df.select("seller_id", "seller_zip_code_prefix", "seller_state")),
          "seller_id", "left")
    .withColumn("category", F.coalesce("product_category_name_english",
                                       "product_category_name", F.lit("uncategorised")))
    .withColumn("volume_cm3", F.col("product_length_cm") * F.col("product_height_cm")
                              * F.col("product_width_cm"))
)

# The 'main' item: highest price, ties broken by item sequence number
main_item_window = Window.partitionBy("order_id").orderBy(F.desc("price"), F.asc("order_item_id"))

main_item_df = (
    items_enriched_df
    .withColumn("rn", F.row_number().over(main_item_window))
    .filter(F.col("rn") == 1)
    .select("order_id",
            F.col("category").alias("main_category"),
            F.col("seller_state").alias("main_seller_state"),
            F.col("seller_zip_code_prefix").alias("main_seller_zip"),
            F.col("product_photos_qty").alias("main_product_photos"),
            F.col("shipping_limit_date"))
)

basket_df = (
    items_enriched_df
    .groupBy("order_id")
    .agg(F.count("*").alias("n_items"),
         F.countDistinct("seller_id").alias("n_sellers"),
         F.round(F.sum("price"), 2).alias("total_price"),
         F.round(F.sum("freight_value"), 2).alias("total_freight"),
         F.sum("product_weight_g").alias("total_weight_g"),
         F.sum("volume_cm3").alias("total_volume_cm3"))
    .join(main_item_df, "order_id")
)

In [12]:
# ---------------------------------------------------------------
# Payments -> one row per order
# ---------------------------------------------------------------
# An order can be paid with several records (e.g. a voucher plus a card).
# payment_type = the method that paid the largest share of the order.
main_payment_window = Window.partitionBy("order_id").orderBy(F.desc("payment_value"),
                                                             F.asc("payment_sequential"))
payments_order_df = (
    order_payments_df
    .withColumn("rn", F.row_number().over(main_payment_window))
    .groupBy("order_id")
    .agg(F.max(F.when(F.col("rn") == 1, F.col("payment_type"))).alias("payment_type"),
         F.max("payment_installments").alias("payment_installments"),
         F.countDistinct("payment_type").alias("n_payment_methods"),
         F.round(F.sum("payment_value"), 2).alias("payment_value"))
)

In [13]:
# ---------------------------------------------------------------
# Assemble the order-level record
# ---------------------------------------------------------------
reviews_per_order_df = (order_reviews_df.groupBy("order_id")
                        .agg(F.avg("review_score").alias("review_score")))   # as in A1

cust_geo = zip_centroid_df.select(F.col("zip_prefix").alias("customer_zip_code_prefix"),
                                  F.col("lat").alias("cust_lat"), F.col("lng").alias("cust_lng"))
sell_geo = zip_centroid_df.select(F.col("zip_prefix").alias("main_seller_zip"),
                                  F.col("lat").alias("sell_lat"), F.col("lng").alias("sell_lng"))

def hours_between(later, earlier):
    '''Difference between two timestamps in hours (double).'''
    return (F.unix_timestamp(later) - F.unix_timestamp(earlier)) / 3600.0

def haversine_km(lat1, lng1, lat2, lng2):
    '''Great-circle distance in km between two (lat, lng) points given in degrees.'''
    dlat = F.radians(lat2 - lat1)
    dlng = F.radians(lng2 - lng1)
    a = (F.sin(dlat / 2) ** 2 +
         F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.sin(dlng / 2) ** 2)
    return 2 * 6371.0 * F.asin(F.sqrt(a))

orders_ml_df = (
    orders_pop_df
    .join(customers_df.select("customer_id", "customer_state", "customer_zip_code_prefix"),
          "customer_id")
    .join(basket_df, "order_id")                       # inner: drops orders with no items
    .join(payments_order_df, "order_id", "left")
    .join(reviews_per_order_df, "order_id", "left")
    .join(F.broadcast(cust_geo), "customer_zip_code_prefix", "left")
    .join(F.broadcast(sell_geo), "main_seller_zip", "left")
    # ---- inputs known at purchase / approval ----
    .withColumn("estimated_delivery_days",
                F.round(hours_between("order_estimated_delivery_date",
                                      "order_purchase_timestamp") / 24, 2))
    .withColumn("shipping_limit_days",
                F.round(hours_between("shipping_limit_date", "order_purchase_timestamp") / 24, 2))
    .withColumn("approval_hours",
                F.round(hours_between("order_approved_at", "order_purchase_timestamp"), 2))
    .withColumn("distance_km",
                F.round(haversine_km(F.col("cust_lat"), F.col("cust_lng"),
                                     F.col("sell_lat"), F.col("sell_lng")), 1))
    .withColumn("same_state", (F.col("customer_state") == F.col("main_seller_state")).cast("int"))
    # ---- outcomes (labels / evaluation only, never model inputs) ----
    .withColumn("delivery_days",
                F.round(hours_between("order_delivered_customer_date",
                                      "order_purchase_timestamp") / 24, 2))
    # Same definition as A1: whole calendar days after the estimated date
    .withColumn("delay_days", F.datediff(F.to_date("order_delivered_customer_date"),
                                         F.to_date("order_estimated_delivery_date")))
    .withColumn("is_late", (F.col("delay_days") > 0).cast("int"))
    .select(
        "order_id", "order_purchase_timestamp",
        # inputs
        "customer_state", "main_seller_state", "same_state", "distance_km",
        "main_category", "main_product_photos",
        "n_items", "n_sellers", "total_price", "total_freight",
        "total_weight_g", "total_volume_cm3",
        "payment_type", "payment_installments", "n_payment_methods", "payment_value",
        "estimated_delivery_days", "shipping_limit_days", "approval_hours",
        # outcomes
        "delivery_days", "delay_days", "is_late", "review_score",
    )
)
orders_ml_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- main_seller_state: string (nullable = true)
 |-- same_state: integer (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- main_category: string (nullable = false)
 |-- main_product_photos: integer (nullable = true)
 |-- n_items: long (nullable = false)
 |-- n_sellers: long (nullable = false)
 |-- total_price: double (nullable = true)
 |-- total_freight: double (nullable = true)
 |-- total_weight_g: double (nullable = true)
 |-- total_volume_cm3: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- n_payment_methods: long (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- estimated_delivery_days: double (nullable = true)
 |-- shipping_limit_days: double (nullable = true)
 |-- approval_hours: double (nullable = true)
 |-- delivery_da

In [14]:
# ---------------------------------------------------------------
# Checks on the order-level record
# ---------------------------------------------------------------
n_orders_ml = orders_ml_df.count()
n_distinct  = orders_ml_df.select("order_id").distinct().count()
checks = [
    ("Orders in population (delivered, 2017 Q1 - 2018 Q3)", orders_pop_df.count()),
    ("Rows in order-level record", n_orders_ml),
    ("Distinct order_id (must equal rows: one row per order)", n_distinct),
]
display(pd.DataFrame(checks, columns=["Check", "Value"]))
assert n_orders_ml == n_distinct, "Duplicate orders in the order-level record"

# Nulls per column (only columns with at least one null are shown)
counts = orders_ml_df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in orders_ml_df.columns]).first()
nulls = [(c, counts[c], round(100 * counts[c] / n_orders_ml, 2))
         for c in orders_ml_df.columns if counts[c]]
pd.DataFrame(nulls, columns=["Column", "Null count", "% of rows"])

,Check,Value
0,"Orders in population (delivered, 2017 Q1 - 201...",96203
1,Rows in order-level record,96203
2,Distinct order_id (must equal rows: one row pe...,96203


,Column,Null count,% of rows
0,distance_km,477,0.50
1,main_product_photos,1356,1.41
2,total_weight_g,16,0.02
3,total_volume_cm3,16,0.02
4,approval_hours,14,0.01
5,review_score,643,0.67


### Train/stream split

**Outputs:**

| File | Rows | Used by |
|---|---|---|
| `data/orders_ml.parquet` | all orders | Input to the split (a deterministic snapshot) |
| `data/train_data.parquet` | ≈70% | Part A: model training and evaluation |
| `data/stream_data.parquet` | ≈30% | Part B: replayed through Kafka by `producer.py`, not used in training |

In [15]:
# ---------------------------------------------------------------
# Deterministic snapshot, then the 70/30 split
# ---------------------------------------------------------------
SEED = 42
SPLIT_WEIGHTS = [0.7, 0.3]

SNAPSHOT_PATH = os.path.join(DATA_DIR, "orders_ml.parquet")
TRAIN_PATH    = os.path.join(DATA_DIR, "train_data.parquet")
STREAM_PATH   = os.path.join(DATA_DIR, "stream_data.parquet")

# 1. One file, sorted by order_id -> fixed layout for randomSplit
(orders_ml_df
 .orderBy("order_id")
 .coalesce(1)
 .write.mode("overwrite").parquet(SNAPSHOT_PATH))

def read_snapshot():
    '''Read the snapshot as ONE partition, so the row layout (and therefore
    the randomSplit result) does not depend on the machine's core count.'''
    return spark.read.parquet(SNAPSHOT_PATH).coalesce(1)

snapshot_df = read_snapshot()
print(f"Snapshot partitions: {snapshot_df.rdd.getNumPartitions()}")

# 2. The split required by the brief
train_df, stream_df = snapshot_df.randomSplit(SPLIT_WEIGHTS, seed=SEED)

# 3. Save both subsets. The stream subset is sorted by purchase time so the
#    producer replays orders in the order they were originally placed.
train_df.write.mode("overwrite").parquet(TRAIN_PATH)
(stream_df
 .orderBy("order_purchase_timestamp", "order_id")
 .coalesce(1)
 .write.mode("overwrite").parquet(STREAM_PATH))

train_df  = spark.read.parquet(TRAIN_PATH).cache()
stream_df = spark.read.parquet(STREAM_PATH)
print(f"Saved {TRAIN_PATH} and {STREAM_PATH}")

Snapshot partitions: 1
Saved data/train_data.parquet and data/stream_data.parquet


In [16]:
# ---------------------------------------------------------------
# Verify the split
# ---------------------------------------------------------------
n_total, n_train, n_stream = snapshot_df.count(), train_df.count(), stream_df.count()
overlap = train_df.select("order_id").intersect(stream_df.select("order_id")).count()
covered = train_df.select("order_id").union(stream_df.select("order_id")).distinct().count()

split_checks = [
    ("Orders in snapshot", f"{n_total:,}"),
    ("Training subset", f"{n_train:,} ({100 * n_train / n_total:.2f}%)"),
    ("Streaming subset", f"{n_stream:,} ({100 * n_stream / n_total:.2f}%)"),
    ("Brief: streaming subset >= 20%", "met" if n_stream / n_total >= 0.20 else "NOT met"),
    ("Orders in both subsets (must be 0)", overlap),
    ("Orders covered by train + stream (must equal snapshot)", f"{covered:,}"),
]
display(pd.DataFrame(split_checks, columns=["Check", "Result"]))
assert overlap == 0 and covered == n_total

# Reproducibility: repeat the split from the saved snapshot and compare
train_again, _ = read_snapshot().randomSplit(SPLIT_WEIGHTS, seed=SEED)
same = (train_again.select("order_id").exceptAll(train_df.select("order_id")).count() == 0
        and train_again.count() == n_train)
print(f"Re-running randomSplit(seed={SEED}) on the snapshot gives the identical training set: {same}")

,Check,Result
0,Orders in snapshot,"96,203"
1,Training subset,"67,274 (69.93%)"
2,Streaming subset,"28,929 (30.07%)"
3,Brief: streaming subset >= 20%,met
4,Orders in both subsets (must be 0),0
5,Orders covered by train + stream (must equal s...,"96,203"


Re-running randomSplit(seed=42) on the snapshot gives the identical training set: True


In [17]:
# ---------------------------------------------------------------
# Is the stream subset representative of the training subset?
# ---------------------------------------------------------------
def profile(df, name):
    return (df.agg(
                F.count("*").alias("orders"),
                F.round(100 * F.avg("is_late"), 2).alias("late_%"),
                F.round(F.avg("review_score"), 3).alias("avg_review"),
                F.round(100 * F.avg((F.col("review_score") <= 2).cast("int")), 2).alias("review_1_2_%"),
                F.round(F.avg("total_price"), 2).alias("avg_total_price"),
                F.round(F.avg("delivery_days"), 2).alias("avg_delivery_days"),
                F.round(100 * F.avg((F.col("customer_state") == "SP").cast("int")), 2).alias("SP_%"),
                F.min("order_purchase_timestamp").alias("first_purchase"),
                F.max("order_purchase_timestamp").alias("last_purchase"))
            .withColumn("subset", F.lit(name)))

profile_pd = (profile(train_df, "train").unionByName(profile(stream_df, "stream"))
              .toPandas().set_index("subset").T)
profile_pd

subset,train,stream
orders,67274,28929
late_%,6.73,6.93
avg_review,4.155,4.16
review_1_2_%,12.75,12.78
avg_total_price,136.34,138.54
avg_delivery_days,12.54,12.53
SP_%,42.22,41.48
first_purchase,2017-01-05 11:56:06,2017-01-05 12:14:58
last_purchase,2018-08-29 15:00:37,2018-08-29 14:52:00
